# Per-dataset output kind

Earthdata is the first earthlens backend whose `OUTPUT_KIND` is resolved **per request** from the catalog row, not fixed on the class. This notebook shows the catalog side of that (no network / credentials needed). For the live download see [Usage](../../reference/earthdata/usage.md).

In [ ]:
from earthlens.earthdata import Catalog

cat = Catalog()
raster = [k for k, d in cat.datasets.items() if d.output_kind == 'raster']
vector = [k for k, d in cat.datasets.items() if d.output_kind == 'vector']
print('raster datasets:', len(raster))
print('vector datasets:', sorted(vector))

## What the output kind controls

When you build `EarthLens(data_source='earthdata', ...)`, the resolved row's `output_kind` becomes the instance's `OUTPUT_KIND`. The facade reads it at `download()` time to gate `aggregate=`:

- a **raster** instance forwards `aggregate=AggregationConfig(...)` into the backend, which reduces a stack of single-timestep granules per window;
- a **vector** / **tabular** instance rejects `aggregate=` with `NotImplementedError`.

A single request must not mix output kinds — `EarthData` rejects that at construction, since one instance carries exactly one kind.

## DAAC registry

Each CMR provider code maps to a DAAC, its cloud region, and the S3 credentials endpoint used for in-region streaming:

In [ ]:
for code in sorted(cat.daacs)[:5]:
    daac = cat.get_daac(code)
    print(f'{code:12s} -> {daac.daac:10s} ({daac.cloud_region})')